# Process Nanopore data

This notebook processes the Nanopore data and generates a table of beta values, similar to the Illumina Methylation arrays.

The data is stored in /Volumes/bronze/methylation/geo_datasets/GSE185307. This contains several files - the ones marked as "grouped" contain the data already processed in the form we need for this.

In [0]:
#!mv "/Volumes/bronze/methylation/geo_datasets/index.html?acc=GSE185307&format=file" /Volumes/bronze/methylation/geo_datasets/GSE185307_RAW.tar

In [0]:
!du -sh  "/Volumes/bronze/methylation/geo_datasets/GSE185307_RAW.tar"

In [0]:
#!mkdir -p /Volumes/bronze/methylation/geo_datasets/GSE185307
#!tar -xvf "/Volumes/bronze/methylation/geo_datasets/GSE185307_RAW.tar" -C  /Volumes/bronze/methylation/geo_datasets/GSE185307

In [0]:
from pyspark.sql import SparkSession
import pandas as pd
import glob
import os

# Set the directory containing all bedgraph.gz files
# Adjust this path as needed
input_path = "/Volumes/bronze/methylation/geo_datasets/GSE185307/"
pattern = "*.grouped.0based.bedgraph.gz"

# Find all matching files
bedgraph_files = glob.glob(os.path.join(input_path, pattern))

bedgraph_files

In [0]:
# Process each file one at a time
for file_path in bedgraph_files:
    # Extract sample name from filename
    filename = os.path.basename(file_path)
    sample = filename.split(".")[0]
    
    # Read the file into Spark
    df = spark.read.option("sep", "\t").csv(f"file://{file_path}")
    
    # Rename columns and build cpg ID
    df = df.withColumnRenamed("_c0", "chrom") \
           .withColumnRenamed("_c1", "start") \
           .withColumnRenamed("_c2", "end") \
           .withColumnRenamed("_c3", sample)
    
    df = df.withColumn("cpg", F.concat_ws(":", F.col("chrom"), F.col("start").cast("string"))) \
           .select("cpg", sample)
    
    # Join into merged DataFrame
    if merged_df is None:
        merged_df = df
    else:
        merged_df = merged_df.join(df, on="cpg", how="outer")

# Write final wide table
merged_df.write.format("delta").mode("overwrite").saveAsTable("bronze.methylation.GSE185307_raw")

print("✅ Wide-format Delta table written to bronze.methylation.GSE185307_raw")

In [0]:
df.head()

In [0]:
methylation_data

In [0]:
%sql

SELECT * FROM bronze.methylation.GSE185307_raw LIMIT 10